# Prompt Viewer

This notebook shows one `llm_prompt` per domain using:
- `demographics+previous_experiences+privacy_attitudes` for all domains except LLM chatbot and LLM healthcare consultation
- `privacy_experiences+privacy_attitudes` for LLM chatbot and LLM healthcare consultation (falls back to `previous_experiences+privacy_attitudes` if needed in the CSV)

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
csv_path = Path('../data/dataset_with_prompts.csv')
df = pd.read_csv(csv_path)

default_prompt_type = 'demographics+previous_experiences+privacy_attitudes'
alternate_prompt_type_requested = 'privacy_experiences+privacy_attitudes'
alternate_prompt_type_fallback = 'previous_experiences+privacy_attitudes'
alternate_domains = {'llm chatbot', 'llm healthcare consultation'}

domain_normalized = df['domain'].astype(str).str.strip().str.lower()

default_rows = df[
    (df['prompt_type'] == default_prompt_type)
    & (~domain_normalized.isin(alternate_domains))
]

alternate_rows = df[
    (df['prompt_type'] == alternate_prompt_type_requested)
    & (domain_normalized.isin(alternate_domains))
]

selected_alternate_prompt_type = alternate_prompt_type_requested
if alternate_rows.empty:
    alternate_rows = df[
        (df['prompt_type'] == alternate_prompt_type_fallback)
        & (domain_normalized.isin(alternate_domains))
    ]
    selected_alternate_prompt_type = alternate_prompt_type_fallback

if default_rows.empty and alternate_rows.empty:
    raise ValueError('No rows found for either requested prompt_type selection.')

one_per_default_domain = (
    default_rows
    .sort_values(['domain', 'prompt_id'])
    .groupby('domain', as_index=False)
    .head(1)
)

one_per_alternate_domain = (
    alternate_rows
    .sort_values(['domain', 'prompt_id'])
    .groupby('domain', as_index=False)
    .head(1)
)

one_per_domain = (
    pd.concat([one_per_default_domain, one_per_alternate_domain], ignore_index=True)
    .loc[:, ['domain', 'prompt_id', 'prompt_type', 'llm_prompt']]
    .sort_values('domain')
    .reset_index(drop=True)
)

print(f"Domains found: {one_per_domain['domain'].nunique()}")
print(f"Alternate domains prompt_type used: {selected_alternate_prompt_type}")
one_per_domain[['domain', 'prompt_id', 'prompt_type']]

Domains found: 5
Alternate domains prompt_type used: previous_experiences+privacy_attitudes


,domain,prompt_id,prompt_type
0,AI agent permissions,808,demographics+previous_experiences+privacy_atti...
1,Chatbot group chat,3208,demographics+previous_experiences+privacy_atti...
2,LLM chatbot,2404,previous_experiences+privacy_attitudes
3,LLM conversational agents,4808,demographics+previous_experiences+privacy_atti...
4,LLM healthcare consultation,4,previous_experiences+privacy_attitudes


In [3]:
for _, row in one_per_domain.iterrows():
    print('=' * 100)
    print(f"Domain: {row['domain']} (prompt_id={row['prompt_id']})")
    print('-' * 100)
    print(row['llm_prompt'])
    print()

Domain: AI agent permissions (prompt_id=808)
----------------------------------------------------------------------------------------------------
ROLE
You are an expert AI simulator acting as a specific human participant in a research study. Your task is to realistically answer a questionnaire about data-sharing scenarios exactly as this specific person would.

USER PRIVACY PERSONA
Internalize the user's traits and answer each scenario as this specific person would. Draw on whichever aspects of their profile are most relevant to the particular scenario.
- Domain Context: AI agent permissions
- Demographics: {'education': "Master's", 'age': '18-24', 'gender': 'Female'}
- Previous AI Experiences: {'On a scale from 1 (Not at all familiar) to 5 (Very familiar), how familiar are you with existing AI tools, such as ChatGPT and Gemini, in terms of your knowledge and usage? [1. Not at all familiar, 2. Slightly familiar, 3. Somewhat familiar, 4. Moderately familiar, 5. Very familiar]': '3', 'Ho